In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report

# 1. Load dataset
df = pd.read_csv("loan_data.csv")

print("First 5 rows:")
print(df.head())

# 2. Handle missing values
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# 3. Remove unnecessary ID column
df = df.drop("Loan_ID", axis=1)

# 4. Convert categorical data into numerical data
df = pd.get_dummies(
    df,
    columns=[
        "Gender", "Married", "Dependents", "Education",
        "Self_Employed", "Property_Area"
    ],
    drop_first=True,
    dtype=int
)

# 5. Separate features and target
X = df.drop("Loan_Status", axis=1)
y = df["Loan_Status"]

# 6. Fixed train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)
# Gini: Measures how mixed the classes are in a node. Lower impurity is preferred.

# Entropy: Measures uncertainty or disorder in a node. Lower entropy is preferred.

# 7. Train Decision Tree using Gini
gini_model = DecisionTreeClassifier(
    criterion="gini", max_depth=3, random_state=42
)
gini_model.fit(X_train, y_train)

# 8. Train Decision Tree using Entropy
entropy_model = DecisionTreeClassifier(
    criterion="entropy", max_depth=3, random_state=42
)
entropy_model.fit(X_train, y_train)

# 9. Compare test-set performance
gini_pred = gini_model.predict(X_test)
entropy_pred = entropy_model.predict(X_test)

print("\nGini Accuracy:",
      round(accuracy_score(y_test, gini_pred), 4))

print("Entropy Accuracy:",
      round(accuracy_score(y_test, entropy_pred), 4))

print("\nGini Classification Report:")
print(classification_report(y_test, gini_pred, zero_division=0))

print("Entropy Classification Report:")
print(classification_report(y_test, entropy_pred, zero_division=0))

# 10. Compare tree structures
print("Gini tree depth:", gini_model.get_depth())
print("Gini number of leaves:", gini_model.get_n_leaves())

print("Entropy tree depth:", entropy_model.get_depth())
print("Entropy number of leaves:", entropy_model.get_n_leaves())

plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
plot_tree(
    gini_model,
    feature_names=X.columns,
    class_names=gini_model.classes_,
    filled=True,
    rounded=True
)
plt.title("Decision Tree - Gini")

plt.subplot(1, 2, 2)
plot_tree(
    entropy_model,
    feature_names=X.columns,
    class_names=entropy_model.classes_,
    filled=True,
    rounded=True
)
plt.title("Decision Tree - Entropy")

plt.tight_layout()
plt.show()

# 11. Tune hyperparameters using cross-validation
param_grid = {
    "max_depth": [2, 3, 4, 5, None],
    "min_samples_leaf": [1, 2, 5, 10]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(criterion="gini", random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train, y_train)

# 12. Display best parameters and score
print("\nBest Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:",
      round(grid_search.best_score_, 4))

# 13. Evaluate optimized model on test data
best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test)

print("Optimized Test Accuracy:",
      round(accuracy_score(y_test, best_pred), 4))

print("\nOptimized Classification Report:")
print(classification_report(y_test, best_pred, zero_division=0))
